In [0]:
library(dataiku)
library(stats) # need this to calculate Mahalanobis Distance
library(parallel) # parallelize
library(dplyr)
library(FNN)
library(cluster)
library(ggplot2)

In [0]:
# Recipe inputs
counterfactual_test_data <- dkuReadDataset("counterfactual_test_data", samplingMethod="head", nbRows=100000)

# path to hardle models and functions
hurdle_predictions_testing <- dkuManagedFolderPath("5NPBmWH1")


# read all models and functions as a list

# List all .rds files in the folder
rds_files <- list.files(hurdle_components_path, pattern = "\\.rds$", full.names = TRUE)

# Read all .rds files into a list
models_n_functions_list <- lapply(rds_files, readRDS)

# Print the names of the loaded objects
names(models_n_functions_list) <- basename(rds_files)

# Display the list
print(models_n_functions_list)

In [0]:
# sanity check: is the read hurdle_function an actual function?
class(models_n_functions_list$hurdle_function)

In [0]:
# get unique municipality observations
mun_properties  <- counterfactual_test_data %>%
    distinct(Mun_Code,
             blue_ss_frac,
             blue_ls_frac,
             red_ls_frac,
             orange_ls_frac,
             yellow_ss_frac,
             red_ss_frac,
             orange_ss_frac,
             yellow_ls_frac,
             roof_strong_wall_strong,
             roof_strong_wall_light,
             roof_strong_wall_salv,
             roof_light_wall_strong,
             roof_light_wall_light,
             roof_light_wall_salv,
             roof_salv_wall_strong,
             roof_salv_wall_light,
             roof_salv_wall_salv,
             island_groups,
             .keep_all = FALSE)

In [0]:
# variables I'm interested in for matching:
match_vars  <- c('blue_ss_frac',
                    'blue_ls_frac',
                    'red_ls_frac',
                    'orange_ls_frac',
                    'yellow_ss_frac',
                    'red_ss_frac',
                    'orange_ss_frac',
                    'yellow_ls_frac',
                    'roof_strong_wall_strong',
                    'roof_strong_wall_light',
                    'roof_strong_wall_salv',
                    'roof_light_wall_strong',
                    'roof_light_wall_light',
                    'roof_light_wall_salv',
                    'roof_salv_wall_strong',
                    'roof_salv_wall_light',
                    'roof_salv_wall_salv'
                   )

In [0]:
# Normalize the variables using z-score
mun_scaled <- mun_properties %>%
  mutate(across(c(blue_ss_frac:roof_salv_wall_salv), scale))

In [0]:
# Recipe outputs
matching_counterfactuals <- dkuManagedFolderPath("C0QXMnX7")